In [1]:
# KCB Loan Portfolio Risk Analyzer
# Phase 2: SQL Analytics

import pandas as pd
import numpy as np
import sqlite3

DB_PATH = 'kcb_portfolio.db'

def run_query(sql, db_path=DB_PATH):
    """
    Reusable query runner.
    Equivalent to a database connection utility in a production environment.
    """
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(sql, conn)
    conn.close()
    return df

# Verify connection
test = run_query("SELECT COUNT(*) AS total_loans FROM loans")
print(f"Database connected ✓  |  Total loans: {test['total_loans'][0]:,}")

Database connected ✓  |  Total loans: 5,000


In [3]:
# QUERY 1: PORTFOLIO BREAKDOWN BY PRODUCT
# This is the first question any credit manager asks:
# What does our book look like by product line?

sql_product_summary = """
    SELECT
        product,
        COUNT(*)                                        AS total_loans,
        ROUND(SUM(loan_amount_kes) / 1e6, 2)           AS total_exposure_kes_m,
        ROUND(AVG(loan_amount_kes) / 1e3, 1)           AS avg_loan_kes_k,
        ROUND(AVG(interest_rate_pct), 2)                AS avg_rate_pct,
        SUM(is_npl)                                     AS npl_count,
        ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)       AS npl_rate_pct,
        ROUND(
            SUM(CASE WHEN is_npl = 1 THEN loan_amount_kes ELSE 0 END)
            * 100.0 / SUM(loan_amount_kes), 1
        )                                               AS npl_exposure_pct
    FROM loans
    GROUP BY product
    ORDER BY total_exposure_kes_m DESC
"""

df_product = run_query(sql_product_summary)

print("=== PORTFOLIO BREAKDOWN BY PRODUCT ===")
print(df_product.to_string(index=False))

=== PORTFOLIO BREAKDOWN BY PRODUCT ===
          product  total_loans  total_exposure_kes_m  avg_loan_kes_k  avg_rate_pct  npl_count  npl_rate_pct  npl_exposure_pct
         Mortgage          729               7092.10          9728.5         15.51        129          17.7              17.4
    Business Loan         1263               2206.24          1746.8         15.43        166          13.1              14.2
    Asset Finance          507               1490.61          2940.1         15.60         65          12.8              12.6
    Personal Loan         1771                535.88           302.6         15.62        283          16.0              16.2
Agricultural Loan          474                143.69           303.1         15.58         89          18.8              17.4
       Staff Loan          256                143.46           560.4         15.71         31          12.1              10.1


In [4]:
# QUERY 2: BRANCH PERFORMANCE WITH WINDOW FUNCTIONS
# Window functions rank rows relative to each other WITHOUT collapsing the data
# RANK() and NTILE() are standard SQL — identical syntax in Oracle

sql_branch_ranking = """
    WITH branch_stats AS (
        SELECT
            branch,
            COUNT(*)                                    AS total_loans,
            ROUND(SUM(loan_amount_kes) / 1e6, 2)       AS exposure_kes_m,
            ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)   AS npl_rate_pct,
            ROUND(AVG(interest_rate_pct), 2)            AS avg_rate_pct
        FROM loans
        WHERE branch != 'Unknown'
        GROUP BY branch
    )
    SELECT
        branch,
        total_loans,
        exposure_kes_m,
        npl_rate_pct,
        avg_rate_pct,

        -- Rank branches by NPL rate (1 = best performing = lowest NPL)
        RANK() OVER (ORDER BY npl_rate_pct ASC)         AS npl_rank,

        -- Quartile grouping: 1 = best 25%, 4 = worst 25%
        NTILE(4) OVER (ORDER BY npl_rate_pct ASC)       AS npl_quartile,

        -- Each branch's share of total portfolio exposure
        ROUND(
            exposure_kes_m * 100.0 / SUM(exposure_kes_m) OVER(), 1
        )                                               AS portfolio_share_pct

    FROM branch_stats
    ORDER BY npl_rank
"""

df_branches = run_query(sql_branch_ranking)

print("=== BRANCH PERFORMANCE RANKING ===")
print(df_branches.to_string(index=False))

=== BRANCH PERFORMANCE RANKING ===
     branch  total_loans  exposure_kes_m  npl_rate_pct  avg_rate_pct  npl_rank  npl_quartile  portfolio_share_pct
    Mombasa          385         1037.67          12.5         15.66         1             1                  9.3
      Nyeri          390          972.41          13.3         15.53         2             1                  8.7
    Eldoret          425          936.48          13.4         15.57         3             1                  8.4
     Nakuru          375          840.32          14.9         15.62         4             2                  7.5
  Westlands          383          954.59          15.1         15.57         5             2                  8.5
    Garissa          400         1084.69          15.3         15.55         6             2                  9.7
      Thika          415          784.01          15.4         15.53         7             3                  7.0
   Machakos          417         1102.14          15.

In [5]:
# QUERY 3: YEAR-ON-YEAR PORTFOLIO TREND 
# LAG() is one of the most powerful window functions in banking analytics
# It lets you compare each row to the previous period — without a self-join
# Identical syntax in Oracle PL/SQL

sql_yoy_trend = """
    WITH yearly AS (
        SELECT
            year_disbursed                                  AS year,
            COUNT(*)                                        AS total_loans,
            ROUND(SUM(loan_amount_kes) / 1e6, 1)           AS exposure_kes_m,
            ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)       AS npl_rate_pct
        FROM loans
        WHERE year_disbursed IS NOT NULL
        GROUP BY year_disbursed
    ),
    with_lag AS (
        SELECT
            year,
            total_loans,
            exposure_kes_m,
            npl_rate_pct,

            -- Previous year values using LAG()
            LAG(total_loans)    OVER (ORDER BY year)        AS prev_year_loans,
            LAG(exposure_kes_m) OVER (ORDER BY year)        AS prev_year_exposure,
            LAG(npl_rate_pct)   OVER (ORDER BY year)        AS prev_year_npl

        FROM yearly
    )
    SELECT
        year,
        total_loans,
        exposure_kes_m,
        npl_rate_pct,

        -- Year-on-year changes
        ROUND(
            (exposure_kes_m - prev_year_exposure)
            * 100.0 / prev_year_exposure, 1
        )                                                   AS exposure_growth_pct,

        ROUND(npl_rate_pct - prev_year_npl, 1)             AS npl_change_ppts

    FROM with_lag
    ORDER BY year
"""

df_trend = run_query(sql_yoy_trend)

print("=== YEAR-ON-YEAR PORTFOLIO TREND ===")
print(df_trend.to_string(index=False))

=== YEAR-ON-YEAR PORTFOLIO TREND ===
  year  total_loans  exposure_kes_m  npl_rate_pct  exposure_growth_pct  npl_change_ppts
2020.0          970          2340.7          15.6                  NaN              NaN
2021.0          994          2142.9          15.7                 -8.5              0.1
2022.0          969          2528.4          14.0                 18.0             -1.7
2023.0         1004          2312.6          16.7                 -8.5              2.7
2024.0         1034          2231.0          14.0                 -3.5             -2.7


In [6]:
# QUERY 4: RELATIONSHIP MANAGER SCORECARD 
# In a real bank this query runs monthly and goes to the Head of Retail Banking
# It identifies which RMs are booking quality loans vs high-risk loans

sql_rm_scorecard = """
    WITH rm_stats AS (
        SELECT
            relationship_manager,
            COUNT(*)                                        AS loans_managed,
            ROUND(SUM(loan_amount_kes) / 1e6, 1)           AS book_size_kes_m,
            ROUND(AVG(loan_amount_kes) / 1e3, 0)           AS avg_loan_kes_k,
            ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)       AS npl_rate_pct,
            SUM(CASE WHEN loan_status = 'Loss'
                THEN loan_amount_kes ELSE 0 END)            AS loss_exposure_kes
        FROM loans
        GROUP BY relationship_manager
    )
    SELECT
        relationship_manager,
        loans_managed,
        book_size_kes_m,
        avg_loan_kes_k,
        npl_rate_pct,
        ROUND(loss_exposure_kes / 1e6, 2)                  AS loss_exposure_kes_m,

        -- Performance tier based on NPL rate
        CASE
            WHEN npl_rate_pct < 10 THEN 'Green  — Low Risk'
            WHEN npl_rate_pct < 20 THEN 'Amber  — Watch'
            ELSE                        'Red    — High Risk'
        END                                                 AS performance_tier,

        -- Rank within team
        RANK() OVER (ORDER BY npl_rate_pct ASC)            AS team_rank

    FROM rm_stats
    ORDER BY team_rank
"""

df_rm = run_query(sql_rm_scorecard)

print("=== RELATIONSHIP MANAGER SCORECARD ===")
print(df_rm.to_string(index=False))

=== RELATIONSHIP MANAGER SCORECARD ===
relationship_manager  loans_managed  book_size_kes_m  avg_loan_kes_k  npl_rate_pct  loss_exposure_kes_m performance_tier  team_rank
          F. Wanjiku            468            889.1          1900.0          13.9                 8.36   Amber  — Watch          1
           R. Mwangi            472           1184.4          2509.0          14.2                60.47   Amber  — Watch          2
            J. Kamau            462           1057.0          2288.0          14.5                40.03   Amber  — Watch          3
           B. Chebet            559           1391.7          2490.0          14.8                45.09   Amber  — Watch          4
          D. Njoroge            522           1075.2          2060.0          15.1                25.12   Amber  — Watch          5
         S. Kipchoge            484           1148.3          2372.0          15.1                34.55   Amber  — Watch          5
           C. Otieno            503  

In [7]:
# STORED PROCEDURE EQUIVALENTS 
# In Oracle PL/SQL, stored procedures are reusable named blocks of SQL logic
# In Python we replicate this pattern with functions — the concept is identical
# This is the bridge between what you're learning and what the job requires

def get_portfolio_health(branch=None, product=None, year=None):
    """
    Reusable portfolio health report.
    Equivalent to an Oracle stored procedure accepting IN parameters.

    Parameters
    ----------
    branch  : str, optional  — filter by branch name
    product : str, optional  — filter by product type
    year    : int, optional  — filter by disbursement year

    Returns
    -------
    DataFrame with portfolio health metrics
    """
    filters = ["1=1"]  # Base condition — always true, makes AND chaining clean
    if branch:
        filters.append(f"branch = '{branch}'")
    if product:
        filters.append(f"product = '{product}'")
    if year:
        filters.append(f"year_disbursed = {year}")

    where_clause = " AND ".join(filters)

    sql = f"""
        SELECT
            COUNT(*)                                    AS total_loans,
            ROUND(SUM(loan_amount_kes) / 1e6, 2)       AS exposure_kes_m,
            ROUND(AVG(interest_rate_pct), 2)            AS avg_rate_pct,
            ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)   AS npl_rate_pct,
            SUM(CASE WHEN loan_status = 'Performing'
                THEN 1 ELSE 0 END)                      AS performing_count,
            SUM(CASE WHEN loan_status = 'Watch'
                THEN 1 ELSE 0 END)                      AS watch_count,
            SUM(CASE WHEN loan_status = 'Substandard'
                THEN 1 ELSE 0 END)                      AS substandard_count,
            SUM(CASE WHEN loan_status = 'Doubtful'
                THEN 1 ELSE 0 END)                      AS doubtful_count,
            SUM(CASE WHEN loan_status = 'Loss'
                THEN 1 ELSE 0 END)                      AS loss_count
        FROM loans
        WHERE {where_clause}
    """
    return run_query(sql)


# ── TEST THE PROCEDURE ────────────────────────────────────────────────────────

print("=== FULL PORTFOLIO ===")
print(get_portfolio_health().to_string(index=False))

print("\n=== NAIROBI CBD BRANCH ===")
print(get_portfolio_health(branch='Nairobi CBD').to_string(index=False))

print("\n=== MORTGAGE PRODUCT — 2023 ===")
print(get_portfolio_health(product='Mortgage', year=2023).to_string(index=False))

print("\n=== KISUMU BRANCH — BUSINESS LOANS ===")
print(get_portfolio_health(branch='Kisumu', product='Business Loan').to_string(index=False))

=== FULL PORTFOLIO ===
 total_loans  exposure_kes_m  avg_rate_pct  npl_rate_pct  performing_count  watch_count  substandard_count  doubtful_count  loss_count
        5000        11611.98         15.55          15.3              3685          552                325             263         175

=== NAIROBI CBD BRANCH ===
 total_loans  exposure_kes_m  avg_rate_pct  npl_rate_pct  performing_count  watch_count  substandard_count  doubtful_count  loss_count
         377          709.68         15.51          15.9               277           40                 28              21          11

=== MORTGAGE PRODUCT — 2023 ===
 total_loans  exposure_kes_m  avg_rate_pct  npl_rate_pct  performing_count  watch_count  substandard_count  doubtful_count  loss_count
         153         1420.08         15.65          16.3               111           17                  6              12           7

=== KISUMU BRANCH — BUSINESS LOANS ===
 total_loans  exposure_kes_m  avg_rate_pct  npl_rate_pct  performi

In [8]:
# FINAL QUERY: PORTFOLIO vs CBK BENCHMARK 
# This is the executive-level query — the one you present to senior management
# It answers: how does our portfolio compare to the industry?

sql_benchmark = """
    WITH portfolio_yearly AS (
        SELECT
            year_disbursed                                  AS year,
            COUNT(*)                                        AS total_loans,
            ROUND(SUM(loan_amount_kes) / 1e6, 1)           AS exposure_kes_m,
            ROUND(SUM(is_npl) * 100.0 / COUNT(*), 1)       AS portfolio_npl_pct,
            ROUND(AVG(interest_rate_pct), 2)                AS avg_rate_pct
        FROM loans
        WHERE year_disbursed IS NOT NULL
        GROUP BY year_disbursed
    )
    SELECT
        p.year,
        p.total_loans,
        p.exposure_kes_m,
        p.avg_rate_pct,
        p.portfolio_npl_pct,
        c.sector_npl_ratio_pct                              AS cbk_npl_pct,
        ROUND(p.portfolio_npl_pct - c.sector_npl_ratio_pct, 1)
                                                            AS npl_vs_sector,
        CASE
            WHEN p.portfolio_npl_pct < c.sector_npl_ratio_pct
            THEN 'Better than sector'
            ELSE 'Worse than sector'
        END                                                 AS vs_benchmark,
        c.avg_lending_rate_pct                              AS cbk_avg_rate,
        ROUND(p.avg_rate_pct - c.avg_lending_rate_pct, 2)  AS rate_premium
    FROM portfolio_yearly p
    LEFT JOIN cbk_benchmarks c ON p.year = c.year
    ORDER BY p.year
"""

df_benchmark = run_query(sql_benchmark)

print("=== KCB PORTFOLIO vs CBK SECTOR BENCHMARK ===")
print(df_benchmark.to_string(index=False))

=== KCB PORTFOLIO vs CBK SECTOR BENCHMARK ===
  year  total_loans  exposure_kes_m  avg_rate_pct  portfolio_npl_pct  cbk_npl_pct  npl_vs_sector       vs_benchmark  cbk_avg_rate  rate_premium
2020.0          970          2340.7         15.53               15.6         14.1            1.5  Worse than sector         12.03          3.50
2021.0          994          2142.9         15.52               15.7         14.0            1.7  Worse than sector         12.09          3.43
2022.0          969          2528.4         15.51               14.0         14.8           -0.8 Better than sector         12.45          3.06
2023.0         1004          2312.6         15.68               16.7         15.7            1.0  Worse than sector         14.26          1.42
2024.0         1034          2231.0         15.53               14.0         16.4           -2.4 Better than sector         16.08         -0.55


### What Data Is Telling Us
#### Full Portfolio Health 
The book carries KES 11.6 billion in exposure with a 15.3% NPL rate. Of 5,000 loans, 763 are classified as non-performing (Substandard + Doubtful + Loss). That is a material credit risk problem worth investigating (exactly what Phase 3 and 4 will do.)

#### CBK Benchmark - The Most Important Table
##### Observation and What it means
2020–2021: Worse than sector by 1.5–1.7ppts - Portfolio was booking riskier loans than industry peers

2022: Better than sector by 0.8ppts - One good year - worth investigating why

2023: Worse again by 1.0ppt - Deterioration resumed

2024: Better by 2.4ppts - Strongest year - but 2024 loans are young, NPLs may emerge later

Rate premium turns negative in 2024 (-0.55%) - KCB is now lending below the CBK average rate - margin compression

That last point is significant. In 2020 the portfolio earned a 3.5% premium over the market rate. By 2024 it is earning below market. (As analyst - flag this immediately to the CFO.)

#### Stored Procedure Results
In Cell 6 - with one function call you can slice the portfolio any way you want:

- Kisumu Business Loans NPL (13.1%) is better than the full portfolio (15.3%) - Kisumu is performing well

- Nairobi CBD NPL (15.9%) is slightly worse than the portfolio average - worth watching

- Mortgages in 2023 (16.3%) are the worst performing segment that year

These are real analytical findings. (present this in an interview.)
